# Sobel vs Scharr Benchmark (Google Colab)

Use this notebook in Google Colab with GPU runtime enabled.

**Comparisons:**
- CPU Sobel (3x3) vs CPU Scharr
- Naive CUDA Sobel vs Naive CUDA Scharr
- Optimized CUDA Sobel vs Optimized CUDA Scharr

**Benchmark rules:**
- Same 4 image sizes: 128x128, 512x512, 1280x720, 1920x1080
- Same input images for every operator/implementation pair
- Same warm-up policy: 2 warm-up runs before 20 measured runs
- Same output depth and post-processing
- Reported: kernel time, total time, stddev, min, max, Scharr/Sobel ratio


In [ ]:
!nvidia-smi


In [ ]:
from pathlib import Path

Path('include').mkdir(exist_ok=True)
Path('src/common').mkdir(parents=True, exist_ok=True)
Path('src/cpu').mkdir(parents=True, exist_ok=True)
Path('src/cuda').mkdir(parents=True, exist_ok=True)
Path('src/tools').mkdir(parents=True, exist_ok=True)
Path('samples/images').mkdir(parents=True, exist_ok=True)


In [ ]:
%%writefile include/image_io.hpp
#pragma once
#include <cstddef>
#include <cstdint>
#include <string>
#include <vector>

struct GrayImage {
    int width = 0;
    int height = 0;
    std::vector<std::uint8_t> pixels;

    bool empty() const;
    std::size_t size() const;
    std::uint8_t& at(int x, int y);
    const std::uint8_t& at(int x, int y) const;
};

GrayImage makeTestPattern(int width, int height);
GrayImage loadPgm(const std::string& path);
void savePgm(const GrayImage& image, const std::string& path);


In [ ]:
%%writefile src/common/image_io.cpp
#include "image_io.hpp"
#include <cctype>
#include <fstream>
#include <stdexcept>

namespace {
std::string readToken(std::istream& input) {
    std::string token;
    char ch = '\0';
    while (input.get(ch)) {
        if (std::isspace(static_cast<unsigned char>(ch))) continue;
        if (ch == '#') { input.ignore(4096, '\n'); continue; }
        token.push_back(ch);
        break;
    }
    while (input.get(ch)) {
        if (std::isspace(static_cast<unsigned char>(ch))) break;
        token.push_back(ch);
    }
    if (token.empty()) throw std::runtime_error("Unexpected end of file while reading PGM token");
    return token;
}
}

bool GrayImage::empty() const { return pixels.empty() || width <= 0 || height <= 0; }
std::size_t GrayImage::size() const { return pixels.size(); }
std::uint8_t& GrayImage::at(int x, int y) { return pixels[static_cast<std::size_t>(y * width + x)]; }
const std::uint8_t& GrayImage::at(int x, int y) const { return pixels[static_cast<std::size_t>(y * width + x)]; }

GrayImage makeTestPattern(int width, int height) {
    GrayImage image;
    image.width = width;
    image.height = height;
    image.pixels.resize(static_cast<std::size_t>(width * height));
    for (int y = 0; y < height; ++y) {
        for (int x = 0; x < width; ++x) {
            std::uint8_t value = static_cast<std::uint8_t>((x + y) % 256);
            if (x > width / 4 && x < (3 * width) / 4 && y > height / 4 && y < (3 * height) / 4) value = 220;
            image.at(x, y) = value;
        }
    }
    return image;
}

GrayImage loadPgm(const std::string& path) {
    std::ifstream file(path, std::ios::binary);
    if (!file) throw std::runtime_error("Failed to open input image: " + path);
    if (readToken(file) != "P5") throw std::runtime_error("Only binary PGM (P5) is supported");
    GrayImage image;
    image.width = std::stoi(readToken(file));
    image.height = std::stoi(readToken(file));
    const int max_value = std::stoi(readToken(file));
    if (max_value != 255) throw std::runtime_error("Only 8-bit PGM files are supported");
    image.pixels.resize(static_cast<std::size_t>(image.width * image.height));
    file.read(reinterpret_cast<char*>(image.pixels.data()), static_cast<std::streamsize>(image.pixels.size()));
    if (!file) throw std::runtime_error("Failed to read image pixel data");
    return image;
}

void savePgm(const GrayImage& image, const std::string& path) {
    std::ofstream file(path, std::ios::binary);
    if (!file) throw std::runtime_error("Failed to open output image: " + path);
    file << "P5\n" << image.width << ' ' << image.height << "\n255\n";
    file.write(reinterpret_cast<const char*>(image.pixels.data()), static_cast<std::streamsize>(image.pixels.size()));
}


In [ ]:
%%writefile src/cpu/sobel_cpu.cpp
#include <algorithm>
#include <chrono>
#include <cstdint>
#include <exception>
#include <iostream>
#include <string>
#include "image_io.hpp"

namespace {
void printUsage(const char* program_name) {
    std::cout << "Usage:\n  " << program_name << " <input.pgm> <output.pgm>\n";
}
}  // namespace

int main(int argc, char** argv) {
    try {
        if (argc != 3) { printUsage(argv[0]); return 1; }
        const auto total_start = std::chrono::high_resolution_clock::now();
        const GrayImage input = loadPgm(argv[1]);
        GrayImage output;
        output.width = input.width;
        output.height = input.height;
        output.pixels.assign(input.size(), 0);

        const auto kernel_start = std::chrono::high_resolution_clock::now();
        for (int y = 1; y < input.height - 1; ++y) {
            for (int x = 1; x < input.width - 1; ++x) {
                const int gx =
                    -input.at(x - 1, y - 1) + input.at(x + 1, y - 1) +
                    -2 * input.at(x - 1, y) + 2 * input.at(x + 1, y) +
                    -input.at(x - 1, y + 1) + input.at(x + 1, y + 1);
                const int gy =
                    -input.at(x - 1, y - 1) - 2 * input.at(x, y - 1) - input.at(x + 1, y - 1) +
                    input.at(x - 1, y + 1) + 2 * input.at(x, y + 1) + input.at(x + 1, y + 1);
                const int magnitude = std::min(255, std::abs(gx) + std::abs(gy));
                output.at(x, y) = static_cast<std::uint8_t>(magnitude);
            }
        }
        const auto kernel_stop = std::chrono::high_resolution_clock::now();
        savePgm(output, argv[2]);
        const auto total_stop = std::chrono::high_resolution_clock::now();
        const std::chrono::duration<double, std::milli> kernel_ms = kernel_stop - kernel_start;
        const std::chrono::duration<double, std::milli> total_ms = total_stop - total_start;
        std::cout << "CPU Sobel completed" << std::endl;
        std::cout << "Input: " << argv[1] << " (" << input.width << "x" << input.height << ")" << std::endl;
        std::cout << "Output: " << argv[2] << std::endl;
        std::cout << "Kernel time: " << kernel_ms.count() << " ms" << std::endl;
        std::cout << "Total path time: " << total_ms.count() << " ms" << std::endl;
        return 0;
    } catch (const std::exception& ex) {
        std::cerr << "Error: " << ex.what() << std::endl;
        return 1;
    }
}


In [ ]:
%%writefile src/cpu/scharr_cpu.cpp
#include <algorithm>
#include <chrono>
#include <cstdint>
#include <cstdlib>
#include <exception>
#include <iostream>
#include <string>
#include "image_io.hpp"

namespace {
void printUsage(const char* program_name) {
    std::cout << "Usage:\n  " << program_name << " <input.pgm> <output.pgm>\n";
}
}  // namespace

int main(int argc, char** argv) {
    try {
        if (argc != 3) { printUsage(argv[0]); return 1; }
        const auto total_start = std::chrono::high_resolution_clock::now();
        const GrayImage input = loadPgm(argv[1]);
        GrayImage output;
        output.width = input.width;
        output.height = input.height;
        output.pixels.assign(input.size(), 0);

        const auto kernel_start = std::chrono::high_resolution_clock::now();
        for (int y = 1; y < input.height - 1; ++y) {
            for (int x = 1; x < input.width - 1; ++x) {
                const int gx =
                    -3 * input.at(x - 1, y - 1) + 3 * input.at(x + 1, y - 1) +
                    -10 * input.at(x - 1, y) + 10 * input.at(x + 1, y) +
                    -3 * input.at(x - 1, y + 1) + 3 * input.at(x + 1, y + 1);
                const int gy =
                    -3 * input.at(x - 1, y - 1) - 10 * input.at(x, y - 1) - 3 * input.at(x + 1, y - 1) +
                    3 * input.at(x - 1, y + 1) + 10 * input.at(x, y + 1) + 3 * input.at(x + 1, y + 1);
                const int magnitude = std::min(255, std::abs(gx) + std::abs(gy));
                output.at(x, y) = static_cast<std::uint8_t>(magnitude);
            }
        }
        const auto kernel_stop = std::chrono::high_resolution_clock::now();
        savePgm(output, argv[2]);
        const auto total_stop = std::chrono::high_resolution_clock::now();
        const std::chrono::duration<double, std::milli> kernel_ms = kernel_stop - kernel_start;
        const std::chrono::duration<double, std::milli> total_ms = total_stop - total_start;
        std::cout << "CPU Scharr completed" << std::endl;
        std::cout << "Input: " << argv[1] << " (" << input.width << "x" << input.height << ")" << std::endl;
        std::cout << "Output: " << argv[2] << std::endl;
        std::cout << "Kernel time: " << kernel_ms.count() << " ms" << std::endl;
        std::cout << "Total path time: " << total_ms.count() << " ms" << std::endl;
        return 0;
    } catch (const std::exception& ex) {
        std::cerr << "Error: " << ex.what() << std::endl;
        return 1;
    }
}


In [ ]:
%%writefile src/cuda/sobel_naive.cu
#include <cstdint>
#include <exception>
#include <iostream>
#include <string>
#include <cuda_runtime.h>
#include "image_io.hpp"

__global__ void sobelNaiveKernel(const std::uint8_t* input, std::uint8_t* output, int width, int height) {
    const int x = blockIdx.x * blockDim.x + threadIdx.x;
    const int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x >= width || y >= height) return;
    const int index = y * width + x;
    if (x == 0 || y == 0 || x == width - 1 || y == height - 1) { output[index] = 0; return; }
    const int gx = -input[(y - 1) * width + (x - 1)] + input[(y - 1) * width + (x + 1)]
                - 2 * input[y * width + (x - 1)] + 2 * input[y * width + (x + 1)]
                - input[(y + 1) * width + (x - 1)] + input[(y + 1) * width + (x + 1)];
    const int gy = -input[(y - 1) * width + (x - 1)] - 2 * input[(y - 1) * width + x] - input[(y - 1) * width + (x + 1)]
                 + input[(y + 1) * width + (x - 1)] + 2 * input[(y + 1) * width + x] + input[(y + 1) * width + (x + 1)];
    const int magnitude = min(255, abs(gx) + abs(gy));
    output[index] = static_cast<std::uint8_t>(magnitude);
}

void checkCuda(cudaError_t error, const char* step) {
    if (error != cudaSuccess) throw std::runtime_error(std::string(step) + ": " + cudaGetErrorString(error));
}

int main(int argc, char** argv) {
    try {
        if (argc != 3) { std::cerr << "Usage: ./sobel_naive <input.pgm> <output.pgm>\n"; return 1; }
        const GrayImage input = loadPgm(argv[1]);
        GrayImage output;
        output.width = input.width;
        output.height = input.height;
        output.pixels.assign(input.size(), 0);
        const std::size_t bytes = input.size() * sizeof(std::uint8_t);
        std::uint8_t* device_input = nullptr;
        std::uint8_t* device_output = nullptr;
        cudaEvent_t total_start, total_stop, kernel_start, kernel_stop;
        checkCuda(cudaEventCreate(&total_start), "create total_start");
        checkCuda(cudaEventCreate(&total_stop), "create total_stop");
        checkCuda(cudaEventCreate(&kernel_start), "create kernel_start");
        checkCuda(cudaEventCreate(&kernel_stop), "create kernel_stop");
        checkCuda(cudaEventRecord(total_start), "record total_start");
        checkCuda(cudaMalloc(&device_input, bytes), "malloc input");
        checkCuda(cudaMalloc(&device_output, bytes), "malloc output");
        checkCuda(cudaMemcpy(device_input, input.pixels.data(), bytes, cudaMemcpyHostToDevice), "copy H2D");
        const dim3 block_dim(16, 16);
        const dim3 grid_dim((input.width + 15) / 16, (input.height + 15) / 16);
        checkCuda(cudaEventRecord(kernel_start), "record kernel_start");
        sobelNaiveKernel<<<grid_dim, block_dim>>>(device_input, device_output, input.width, input.height);
        checkCuda(cudaGetLastError(), "kernel launch");
        checkCuda(cudaEventRecord(kernel_stop), "record kernel_stop");
        checkCuda(cudaEventSynchronize(kernel_stop), "sync kernel_stop");
        checkCuda(cudaMemcpy(output.pixels.data(), device_output, bytes, cudaMemcpyDeviceToHost), "copy D2H");
        checkCuda(cudaEventRecord(total_stop), "record total_stop");
        checkCuda(cudaEventSynchronize(total_stop), "sync total_stop");
        float kernel_ms = 0.0f, total_ms = 0.0f;
        checkCuda(cudaEventElapsedTime(&kernel_ms, kernel_start, kernel_stop), "elapsed kernel");
        checkCuda(cudaEventElapsedTime(&total_ms, total_start, total_stop), "elapsed total");
        savePgm(output, argv[2]);
        std::cout << "Naive CUDA Sobel completed" << std::endl;
        std::cout << "Input: " << argv[1] << " (" << input.width << "x" << input.height << ")" << std::endl;
        std::cout << "Output: " << argv[2] << std::endl;
        std::cout << "Kernel time: " << kernel_ms << " ms" << std::endl;
        std::cout << "Total GPU path time: " << total_ms << " ms" << std::endl;
        cudaFree(device_input);
        cudaFree(device_output);
        cudaEventDestroy(total_start);
        cudaEventDestroy(total_stop);
        cudaEventDestroy(kernel_start);
        cudaEventDestroy(kernel_stop);
        return 0;
    } catch (const std::exception& ex) {
        std::cerr << "Error: " << ex.what() << '\n';
        return 1;
    }
}


In [ ]:
%%writefile src/cuda/scharr_naive.cu
#include <cstdint>
#include <exception>
#include <iostream>
#include <string>
#include <cuda_runtime.h>
#include "image_io.hpp"

__global__ void scharrNaiveKernel(const std::uint8_t* input, std::uint8_t* output, int width, int height) {
    const int x = blockIdx.x * blockDim.x + threadIdx.x;
    const int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x >= width || y >= height) return;
    const int index = y * width + x;
    if (x == 0 || y == 0 || x == width - 1 || y == height - 1) { output[index] = 0; return; }
    const int gx = -3 * input[(y - 1) * width + (x - 1)] + 3 * input[(y - 1) * width + (x + 1)]
                 - 10 * input[y * width + (x - 1)] + 10 * input[y * width + (x + 1)]
                 - 3 * input[(y + 1) * width + (x - 1)] + 3 * input[(y + 1) * width + (x + 1)];
    const int gy = -3 * input[(y - 1) * width + (x - 1)] - 10 * input[(y - 1) * width + x] - 3 * input[(y - 1) * width + (x + 1)]
                 + 3 * input[(y + 1) * width + (x - 1)] + 10 * input[(y + 1) * width + x] + 3 * input[(y + 1) * width + (x + 1)];
    const int magnitude = min(255, abs(gx) + abs(gy));
    output[index] = static_cast<std::uint8_t>(magnitude);
}

void checkCuda(cudaError_t error, const char* step) {
    if (error != cudaSuccess) throw std::runtime_error(std::string(step) + ": " + cudaGetErrorString(error));
}

int main(int argc, char** argv) {
    try {
        if (argc != 3) { std::cerr << "Usage: ./scharr_naive <input.pgm> <output.pgm>\n"; return 1; }
        const GrayImage input = loadPgm(argv[1]);
        GrayImage output;
        output.width = input.width;
        output.height = input.height;
        output.pixels.assign(input.size(), 0);
        const std::size_t bytes = input.size() * sizeof(std::uint8_t);
        std::uint8_t* device_input = nullptr;
        std::uint8_t* device_output = nullptr;
        cudaEvent_t total_start, total_stop, kernel_start, kernel_stop;
        checkCuda(cudaEventCreate(&total_start), "create total_start");
        checkCuda(cudaEventCreate(&total_stop), "create total_stop");
        checkCuda(cudaEventCreate(&kernel_start), "create kernel_start");
        checkCuda(cudaEventCreate(&kernel_stop), "create kernel_stop");
        checkCuda(cudaEventRecord(total_start), "record total_start");
        checkCuda(cudaMalloc(&device_input, bytes), "malloc input");
        checkCuda(cudaMalloc(&device_output, bytes), "malloc output");
        checkCuda(cudaMemcpy(device_input, input.pixels.data(), bytes, cudaMemcpyHostToDevice), "copy H2D");
        const dim3 block_dim(16, 16);
        const dim3 grid_dim((input.width + 15) / 16, (input.height + 15) / 16);
        checkCuda(cudaEventRecord(kernel_start), "record kernel_start");
        scharrNaiveKernel<<<grid_dim, block_dim>>>(device_input, device_output, input.width, input.height);
        checkCuda(cudaGetLastError(), "kernel launch");
        checkCuda(cudaEventRecord(kernel_stop), "record kernel_stop");
        checkCuda(cudaEventSynchronize(kernel_stop), "sync kernel_stop");
        checkCuda(cudaMemcpy(output.pixels.data(), device_output, bytes, cudaMemcpyDeviceToHost), "copy D2H");
        checkCuda(cudaEventRecord(total_stop), "record total_stop");
        checkCuda(cudaEventSynchronize(total_stop), "sync total_stop");
        float kernel_ms = 0.0f, total_ms = 0.0f;
        checkCuda(cudaEventElapsedTime(&kernel_ms, kernel_start, kernel_stop), "elapsed kernel");
        checkCuda(cudaEventElapsedTime(&total_ms, total_start, total_stop), "elapsed total");
        savePgm(output, argv[2]);
        std::cout << "Naive CUDA Scharr completed" << std::endl;
        std::cout << "Input: " << argv[1] << " (" << input.width << "x" << input.height << ")" << std::endl;
        std::cout << "Output: " << argv[2] << std::endl;
        std::cout << "Kernel time: " << kernel_ms << " ms" << std::endl;
        std::cout << "Total GPU path time: " << total_ms << " ms" << std::endl;
        cudaFree(device_input);
        cudaFree(device_output);
        cudaEventDestroy(total_start);
        cudaEventDestroy(total_stop);
        cudaEventDestroy(kernel_start);
        cudaEventDestroy(kernel_stop);
        return 0;
    } catch (const std::exception& ex) {
        std::cerr << "Error: " << ex.what() << '\n';
        return 1;
    }
}


In [ ]:
%%writefile src/cuda/sobel_optimized.cu
#include <cstdint>
#include <exception>
#include <iostream>
#include <string>
#include <cuda_runtime.h>
#include "image_io.hpp"

namespace {
constexpr int kBlockX = 16;
constexpr int kBlockY = 16;

__device__ std::uint8_t loadPixelClamped(const std::uint8_t* input, int width, int height, int x, int y) {
    if (x < 0 || y < 0 || x >= width || y >= height) return 0;
    return input[y * width + x];
}

__global__ void sobelOptimizedKernel(const std::uint8_t* input, std::uint8_t* output, int width, int height) {
    __shared__ std::uint8_t tile[kBlockY + 2][kBlockX + 2];
    const int global_x = blockIdx.x * blockDim.x + threadIdx.x;
    const int global_y = blockIdx.y * blockDim.y + threadIdx.y;
    const int local_x = threadIdx.x + 1;
    const int local_y = threadIdx.y + 1;
    tile[local_y][local_x] = loadPixelClamped(input, width, height, global_x, global_y);
    if (threadIdx.x == 0) tile[local_y][0] = loadPixelClamped(input, width, height, global_x - 1, global_y);
    if (threadIdx.x == blockDim.x - 1) tile[local_y][local_x + 1] = loadPixelClamped(input, width, height, global_x + 1, global_y);
    if (threadIdx.y == 0) tile[0][local_x] = loadPixelClamped(input, width, height, global_x, global_y - 1);
    if (threadIdx.y == blockDim.y - 1) tile[local_y + 1][local_x] = loadPixelClamped(input, width, height, global_x, global_y + 1);
    if (threadIdx.x == 0 && threadIdx.y == 0) tile[0][0] = loadPixelClamped(input, width, height, global_x - 1, global_y - 1);
    if (threadIdx.x == blockDim.x - 1 && threadIdx.y == 0) tile[0][local_x + 1] = loadPixelClamped(input, width, height, global_x + 1, global_y - 1);
    if (threadIdx.x == 0 && threadIdx.y == blockDim.y - 1) tile[local_y + 1][0] = loadPixelClamped(input, width, height, global_x - 1, global_y + 1);
    if (threadIdx.x == blockDim.x - 1 && threadIdx.y == blockDim.y - 1) tile[local_y + 1][local_x + 1] = loadPixelClamped(input, width, height, global_x + 1, global_y + 1);
    __syncthreads();
    if (global_x >= width || global_y >= height) return;
    const int index = global_y * width + global_x;
    if (global_x == 0 || global_y == 0 || global_x == width - 1 || global_y == height - 1) { output[index] = 0; return; }
    const int gx = -tile[local_y - 1][local_x - 1] + tile[local_y - 1][local_x + 1]
                - 2 * tile[local_y][local_x - 1] + 2 * tile[local_y][local_x + 1]
                - tile[local_y + 1][local_x - 1] + tile[local_y + 1][local_x + 1];
    const int gy = -tile[local_y - 1][local_x - 1] - 2 * tile[local_y - 1][local_x] - tile[local_y - 1][local_x + 1]
                 + tile[local_y + 1][local_x - 1] + 2 * tile[local_y + 1][local_x] + tile[local_y + 1][local_x + 1];
    const int magnitude = min(255, abs(gx) + abs(gy));
    output[index] = static_cast<std::uint8_t>(magnitude);
}

void checkCuda(cudaError_t error, const char* step) {
    if (error != cudaSuccess) throw std::runtime_error(std::string(step) + ": " + cudaGetErrorString(error));
}
}  // namespace

int main(int argc, char** argv) {
    try {
        if (argc != 3) { std::cerr << "Usage: ./sobel_optimized <input.pgm> <output.pgm>\n"; return 1; }
        const GrayImage input = loadPgm(argv[1]);
        GrayImage output;
        output.width = input.width;
        output.height = input.height;
        output.pixels.assign(input.size(), 0);
        const std::size_t bytes = input.size() * sizeof(std::uint8_t);
        std::uint8_t* device_input = nullptr;
        std::uint8_t* device_output = nullptr;
        cudaEvent_t total_start, total_stop, kernel_start, kernel_stop;
        checkCuda(cudaEventCreate(&total_start), "cudaEventCreate total_start");
        checkCuda(cudaEventCreate(&total_stop), "cudaEventCreate total_stop");
        checkCuda(cudaEventCreate(&kernel_start), "cudaEventCreate kernel_start");
        checkCuda(cudaEventCreate(&kernel_stop), "cudaEventCreate kernel_stop");
        checkCuda(cudaEventRecord(total_start), "cudaEventRecord total_start");
        checkCuda(cudaMalloc(&device_input, bytes), "cudaMalloc device_input");
        checkCuda(cudaMalloc(&device_output, bytes), "cudaMalloc device_output");
        checkCuda(cudaMemcpy(device_input, input.pixels.data(), bytes, cudaMemcpyHostToDevice), "cudaMemcpy H2D");
        const dim3 block_dim(kBlockX, kBlockY);
        const dim3 grid_dim(
            static_cast<unsigned int>((input.width + block_dim.x - 1) / block_dim.x),
            static_cast<unsigned int>((input.height + block_dim.y - 1) / block_dim.y));
        checkCuda(cudaEventRecord(kernel_start), "cudaEventRecord kernel_start");
        sobelOptimizedKernel<<<grid_dim, block_dim>>>(device_input, device_output, input.width, input.height);
        checkCuda(cudaGetLastError(), "sobelOptimizedKernel launch");
        checkCuda(cudaEventRecord(kernel_stop), "cudaEventRecord kernel_stop");
        checkCuda(cudaEventSynchronize(kernel_stop), "cudaEventSynchronize kernel_stop");
        checkCuda(cudaMemcpy(output.pixels.data(), device_output, bytes, cudaMemcpyDeviceToHost), "cudaMemcpy D2H");
        checkCuda(cudaEventRecord(total_stop), "cudaEventRecord total_stop");
        checkCuda(cudaEventSynchronize(total_stop), "cudaEventSynchronize total_stop");
        float kernel_ms = 0.0f, total_ms = 0.0f;
        checkCuda(cudaEventElapsedTime(&kernel_ms, kernel_start, kernel_stop), "cudaEventElapsedTime kernel");
        checkCuda(cudaEventElapsedTime(&total_ms, total_start, total_stop), "cudaEventElapsedTime total");
        savePgm(output, argv[2]);
        std::cout << "Optimized CUDA Sobel completed" << std::endl;
        std::cout << "Input: " << argv[1] << " (" << input.width << "x" << input.height << ")" << std::endl;
        std::cout << "Output: " << argv[2] << std::endl;
        std::cout << "Kernel time: " << kernel_ms << " ms" << std::endl;
        std::cout << "Total GPU path time: " << total_ms << " ms" << std::endl;
        cudaFree(device_input);
        cudaFree(device_output);
        cudaEventDestroy(total_start);
        cudaEventDestroy(total_stop);
        cudaEventDestroy(kernel_start);
        cudaEventDestroy(kernel_stop);
        return 0;
    } catch (const std::exception& ex) {
        std::cerr << "Error: " << ex.what() << std::endl;
        return 1;
    }
}


In [ ]:
%%writefile src/cuda/scharr_optimized.cu
#include <cstdint>
#include <exception>
#include <iostream>
#include <string>
#include <cuda_runtime.h>
#include "image_io.hpp"

namespace {
constexpr int kBlockX = 16;
constexpr int kBlockY = 16;

__device__ std::uint8_t loadPixelClamped(const std::uint8_t* input, int width, int height, int x, int y) {
    if (x < 0 || y < 0 || x >= width || y >= height) return 0;
    return input[y * width + x];
}

__global__ void scharrOptimizedKernel(const std::uint8_t* input, std::uint8_t* output, int width, int height) {
    __shared__ std::uint8_t tile[kBlockY + 2][kBlockX + 2];
    const int global_x = blockIdx.x * blockDim.x + threadIdx.x;
    const int global_y = blockIdx.y * blockDim.y + threadIdx.y;
    const int local_x = threadIdx.x + 1;
    const int local_y = threadIdx.y + 1;
    tile[local_y][local_x] = loadPixelClamped(input, width, height, global_x, global_y);
    if (threadIdx.x == 0) tile[local_y][0] = loadPixelClamped(input, width, height, global_x - 1, global_y);
    if (threadIdx.x == blockDim.x - 1) tile[local_y][local_x + 1] = loadPixelClamped(input, width, height, global_x + 1, global_y);
    if (threadIdx.y == 0) tile[0][local_x] = loadPixelClamped(input, width, height, global_x, global_y - 1);
    if (threadIdx.y == blockDim.y - 1) tile[local_y + 1][local_x] = loadPixelClamped(input, width, height, global_x, global_y + 1);
    if (threadIdx.x == 0 && threadIdx.y == 0) tile[0][0] = loadPixelClamped(input, width, height, global_x - 1, global_y - 1);
    if (threadIdx.x == blockDim.x - 1 && threadIdx.y == 0) tile[0][local_x + 1] = loadPixelClamped(input, width, height, global_x + 1, global_y - 1);
    if (threadIdx.x == 0 && threadIdx.y == blockDim.y - 1) tile[local_y + 1][0] = loadPixelClamped(input, width, height, global_x - 1, global_y + 1);
    if (threadIdx.x == blockDim.x - 1 && threadIdx.y == blockDim.y - 1) tile[local_y + 1][local_x + 1] = loadPixelClamped(input, width, height, global_x + 1, global_y + 1);
    __syncthreads();
    if (global_x >= width || global_y >= height) return;
    const int index = global_y * width + global_x;
    if (global_x == 0 || global_y == 0 || global_x == width - 1 || global_y == height - 1) { output[index] = 0; return; }
    const int gx = -3 * tile[local_y - 1][local_x - 1] + 3 * tile[local_y - 1][local_x + 1]
                - 10 * tile[local_y][local_x - 1] + 10 * tile[local_y][local_x + 1]
                - 3 * tile[local_y + 1][local_x - 1] + 3 * tile[local_y + 1][local_x + 1];
    const int gy = -3 * tile[local_y - 1][local_x - 1] - 10 * tile[local_y - 1][local_x] - 3 * tile[local_y - 1][local_x + 1]
                 + 3 * tile[local_y + 1][local_x - 1] + 10 * tile[local_y + 1][local_x] + 3 * tile[local_y + 1][local_x + 1];
    const int magnitude = min(255, abs(gx) + abs(gy));
    output[index] = static_cast<std::uint8_t>(magnitude);
}

void checkCuda(cudaError_t error, const char* step) {
    if (error != cudaSuccess) throw std::runtime_error(std::string(step) + ": " + cudaGetErrorString(error));
}
}  // namespace

int main(int argc, char** argv) {
    try {
        if (argc != 3) { std::cerr << "Usage: ./scharr_optimized <input.pgm> <output.pgm>\n"; return 1; }
        const GrayImage input = loadPgm(argv[1]);
        GrayImage output;
        output.width = input.width;
        output.height = input.height;
        output.pixels.assign(input.size(), 0);
        const std::size_t bytes = input.size() * sizeof(std::uint8_t);
        std::uint8_t* device_input = nullptr;
        std::uint8_t* device_output = nullptr;
        cudaEvent_t total_start, total_stop, kernel_start, kernel_stop;
        checkCuda(cudaEventCreate(&total_start), "cudaEventCreate total_start");
        checkCuda(cudaEventCreate(&total_stop), "cudaEventCreate total_stop");
        checkCuda(cudaEventCreate(&kernel_start), "cudaEventCreate kernel_start");
        checkCuda(cudaEventCreate(&kernel_stop), "cudaEventCreate kernel_stop");
        checkCuda(cudaEventRecord(total_start), "cudaEventRecord total_start");
        checkCuda(cudaMalloc(&device_input, bytes), "cudaMalloc device_input");
        checkCuda(cudaMalloc(&device_output, bytes), "cudaMalloc device_output");
        checkCuda(cudaMemcpy(device_input, input.pixels.data(), bytes, cudaMemcpyHostToDevice), "cudaMemcpy H2D");
        const dim3 block_dim(kBlockX, kBlockY);
        const dim3 grid_dim(
            static_cast<unsigned int>((input.width + block_dim.x - 1) / block_dim.x),
            static_cast<unsigned int>((input.height + block_dim.y - 1) / block_dim.y));
        checkCuda(cudaEventRecord(kernel_start), "cudaEventRecord kernel_start");
        scharrOptimizedKernel<<<grid_dim, block_dim>>>(device_input, device_output, input.width, input.height);
        checkCuda(cudaGetLastError(), "scharrOptimizedKernel launch");
        checkCuda(cudaEventRecord(kernel_stop), "cudaEventRecord kernel_stop");
        checkCuda(cudaEventSynchronize(kernel_stop), "cudaEventSynchronize kernel_stop");
        checkCuda(cudaMemcpy(output.pixels.data(), device_output, bytes, cudaMemcpyDeviceToHost), "cudaMemcpy D2H");
        checkCuda(cudaEventRecord(total_stop), "cudaEventRecord total_stop");
        checkCuda(cudaEventSynchronize(total_stop), "cudaEventSynchronize total_stop");
        float kernel_ms = 0.0f, total_ms = 0.0f;
        checkCuda(cudaEventElapsedTime(&kernel_ms, kernel_start, kernel_stop), "cudaEventElapsedTime kernel");
        checkCuda(cudaEventElapsedTime(&total_ms, total_start, total_stop), "cudaEventElapsedTime total");
        savePgm(output, argv[2]);
        std::cout << "Optimized CUDA Scharr completed" << std::endl;
        std::cout << "Input: " << argv[1] << " (" << input.width << "x" << input.height << ")" << std::endl;
        std::cout << "Output: " << argv[2] << std::endl;
        std::cout << "Kernel time: " << kernel_ms << " ms" << std::endl;
        std::cout << "Total GPU path time: " << total_ms << " ms" << std::endl;
        cudaFree(device_input);
        cudaFree(device_output);
        cudaEventDestroy(total_start);
        cudaEventDestroy(total_stop);
        cudaEventDestroy(kernel_start);
        cudaEventDestroy(kernel_stop);
        return 0;
    } catch (const std::exception& ex) {
        std::cerr << "Error: " << ex.what() << std::endl;
        return 1;
    }
}


In [ ]:
%%writefile src/tools/image_compare.cpp
#include <cmath>
#include <cstdint>
#include <exception>
#include <iostream>
#include <string>
#include "image_io.hpp"

namespace {
void printUsage(const char* program_name) {
    std::cout << "Usage:\n  " << program_name << " <reference.pgm> <candidate.pgm>\n";
}
}  // namespace

int main(int argc, char** argv) {
    try {
        if (argc != 3) { printUsage(argv[0]); return 1; }
        const GrayImage reference = loadPgm(argv[1]);
        const GrayImage candidate = loadPgm(argv[2]);
        if (reference.width != candidate.width || reference.height != candidate.height) {
            std::cerr << "Error: image dimensions do not match" << std::endl;
            return 1;
        }
        std::size_t different_pixels = 0;
        int max_abs_diff = 0;
        std::uint64_t sum_abs_diff = 0;
        for (std::size_t i = 0; i < reference.pixels.size(); ++i) {
            const int diff = std::abs(static_cast<int>(reference.pixels[i]) - static_cast<int>(candidate.pixels[i]));
            if (diff != 0) ++different_pixels;
            if (diff > max_abs_diff) max_abs_diff = diff;
            sum_abs_diff += static_cast<std::uint64_t>(diff);
        }
        const double mean_abs_diff = reference.pixels.empty() ? 0.0 : static_cast<double>(sum_abs_diff) / static_cast<double>(reference.pixels.size());
        const double different_ratio = reference.pixels.empty() ? 0.0 : static_cast<double>(different_pixels) / static_cast<double>(reference.pixels.size());
        std::cout << "Comparison complete" << std::endl;
        std::cout << "Reference: " << argv[1] << std::endl;
        std::cout << "Candidate: " << argv[2] << std::endl;
        std::cout << "Different pixels: " << different_pixels << std::endl;
        std::cout << "Difference ratio: " << different_ratio << std::endl;
        std::cout << "Max abs diff: " << max_abs_diff << std::endl;
        std::cout << "Mean abs diff: " << mean_abs_diff << std::endl;
        return 0;
    } catch (const std::exception& ex) {
        std::cerr << "Error: " << ex.what() << std::endl;
        return 1;
    }
}


In [ ]:
from pathlib import Path

def write_pgm(path, width, height):
    pixels = bytearray(width * height)
    for y in range(height):
        for x in range(width):
            value = (x + y) % 256
            if x > width // 4 and x < (3 * width) // 4 and y > height // 4 and y < (3 * height) // 4:
                value = 220
            pixels[y * width + x] = value
    with open(path, 'wb') as f:
        f.write(f'P5\n{width} {height}\n255\n'.encode())
        f.write(pixels)

Path('samples/images').mkdir(parents=True, exist_ok=True)
for name, w, h in [
    ('test_128x128.pgm', 128, 128),
    ('test_512x512.pgm', 512, 512),
    ('test_1280x720.pgm', 1280, 720),
    ('test_1920x1080.pgm', 1920, 1080),
]:
    write_pgm(Path('samples/images') / name, w, h)
print('Generated benchmark inputs')


In [ ]:
!python3 generate_pgm.py
!g++ -O2 -std=c++17 -Iinclude src/cpu/sobel_cpu.cpp src/common/image_io.cpp -o sobel_cpu
!g++ -O2 -std=c++17 -Iinclude src/cpu/scharr_cpu.cpp src/common/image_io.cpp -o scharr_cpu
!nvcc -O2 -I. -Iinclude src/cuda/sobel_naive.cu src/common/image_io.cpp -o sobel_naive
!nvcc -O2 -I. -Iinclude src/cuda/scharr_naive.cu src/common/image_io.cpp -o scharr_naive
!nvcc -O2 -I. -Iinclude src/cuda/sobel_optimized.cu src/common/image_io.cpp -o sobel_optimized
!nvcc -O2 -I. -Iinclude src/cuda/scharr_optimized.cu src/common/image_io.cpp -o scharr_optimized
!g++ -O2 -std=c++17 -Iinclude src/tools/image_compare.cpp src/common/image_io.cpp -o compare_images


In [ ]:
!./sobel_cpu samples/images/test_512x512.pgm samples/images/test_512x512_sobel_cpu.pgm
!./scharr_cpu samples/images/test_512x512.pgm samples/images/test_512x512_scharr_cpu.pgm
!./sobel_naive samples/images/test_512x512.pgm samples/images/test_512x512_sobel_naive.pgm
!./scharr_naive samples/images/test_512x512.pgm samples/images/test_512x512_scharr_naive.pgm
!./sobel_optimized samples/images/test_512x512.pgm samples/images/test_512x512_sobel_opt.pgm
!./scharr_optimized samples/images/test_512x512.pgm samples/images/test_512x512_scharr_opt.pgm

print()
print("=== Validation: Sobel CPU vs Sobel Naive CUDA ===")
!./compare_images samples/images/test_512x512_sobel_cpu.pgm samples/images/test_512x512_sobel_naive.pgm
print()
print("=== Validation: Sobel CPU vs Sobel Optimized CUDA ===")
!./compare_images samples/images/test_512x512_sobel_cpu.pgm samples/images/test_512x512_sobel_opt.pgm
print()
print("=== Validation: Scharr CPU vs Scharr Naive CUDA ===")
!./compare_images samples/images/test_512x512_scharr_cpu.pgm samples/images/test_512x512_scharr_naive.pgm
print()
print("=== Validation: Scharr CPU vs Scharr Optimized CUDA ===")
!./compare_images samples/images/test_512x512_scharr_cpu.pgm samples/images/test_512x512_scharr_opt.pgm


In [ ]:
\
import re
import subprocess
import pandas as pd

warmup_runs = 2
measured_runs = 20

images = [
    ('128x128', 128, 128, 'samples/images/test_128x128.pgm'),
    ('512x512', 512, 512, 'samples/images/test_512x512.pgm'),
    ('1280x720', 1280, 720, 'samples/images/test_1280x720.pgm'),
    ('1920x1080', 1920, 1080, 'samples/images/test_1920x1080.pgm'),
]

operators = [
    ('sobel', 'cpu', './sobel_cpu'),
    ('scharr', 'cpu', './scharr_cpu'),
    ('sobel', 'naive_cuda', './sobel_naive'),
    ('scharr', 'naive_cuda', './scharr_naive'),
    ('sobel', 'optimized_cuda', './sobel_optimized'),
    ('scharr', 'optimized_cuda', './scharr_optimized'),
]

kernel_pattern = re.compile(r'Kernel time:\s*([0-9.]+)\s*ms')
total_pattern = re.compile(r'(?:Total GPU path time|Total path time):\s*([0-9.]+)\s*ms')

rows = []

for size_label, width, height, input_path in images:
    pixels = width * height

    for operator, impl, executable in operators:
        kernel_times = []
        total_times = []
        output_path = f'samples/images/{size_label}_{operator}_{impl}.pgm'

        # Warm-up runs
        for _ in range(warmup_runs):
            subprocess.run([executable, input_path, output_path],
                           capture_output=True, text=True, check=True)

        # Measured runs
        for _ in range(measured_runs):
            completed = subprocess.run([executable, input_path, output_path],
                                      capture_output=True, text=True, check=True)
            stdout = completed.stdout
            km = kernel_pattern.search(stdout)
            tm = total_pattern.search(stdout)
            if km is None or tm is None:
                raise RuntimeError(f'Failed to parse timing for {operator} {impl} {size_label}:\n{stdout}')
            kernel_times.append(float(km.group(1)))
            total_times.append(float(tm.group(1)))

        kern_series = pd.Series(kernel_times)
        total_series = pd.Series(total_times)
        rows.append({
            'operator': operator,
            'implementation': impl,
            'size': size_label,
            'width': width,
            'height': height,
            'pixels': pixels,
            'warmup_runs': warmup_runs,
            'measured_runs': measured_runs,
            'avg_kernel_ms': kern_series.mean(),
            'std_kernel_ms': kern_series.std(ddof=0),
            'min_kernel_ms': kern_series.min(),
            'max_kernel_ms': kern_series.max(),
            'avg_total_ms': total_series.mean(),
            'std_total_ms': total_series.std(ddof=0),
            'min_total_ms': total_series.min(),
            'max_total_ms': total_series.max(),
        })

results_df = pd.DataFrame(rows).sort_values(['implementation', 'operator', 'pixels']).reset_index(drop=True)
print("=== Full Results DataFrame ===")
display(results_df)


In [ ]:
\
print("=" * 80)
print("TABLE 1: CPU Sobel vs CPU Scharr")
print("=" * 80)

cpu_df = results_df[results_df['implementation'] == 'cpu'][['size','operator','avg_kernel_ms','std_kernel_ms','min_kernel_ms','max_kernel_ms','avg_total_ms','std_total_ms','min_total_ms','max_total_ms']].copy()

for size_label in ['128x128', '512x512', '1280x720', '1920x1080']:
    sobel_row = cpu_df[(cpu_df['size'] == size_label) & (cpu_df['operator'] == 'sobel')].iloc[0]
    scharr_row = cpu_df[(cpu_df['size'] == size_label) & (cpu_df['operator'] == 'scharr')].iloc[0]
    kernel_ratio = scharr_row['avg_kernel_ms'] / sobel_row['avg_kernel_ms']
    total_ratio = scharr_row['avg_total_ms'] / sobel_row['avg_total_ms']
    print(f"\n{size_label}  (Scharr/Sobel kernel ratio: {kernel_ratio:.4f}, total ratio: {total_ratio:.4f})")
    print(f"{'':>8} {'sobel':>12} {'scharr':>12} {'sobel':>12} {'scharr':>12}")
    print(f"{'':>8} {'kernel ms':>12} {'kernel ms':>12} {'total ms':>12} {'total ms':>12}")
    print(f"{'':>8} {sobel_row['avg_kernel_ms']:>12.5f} {scharr_row['avg_kernel_ms']:>12.5f} {sobel_row['avg_total_ms']:>12.5f} {scharr_row['avg_total_ms']:>12.5f}")
    print(f"{'':>8} {'std: '+str(round(sobel_row['std_kernel_ms'],5)):>12} {'std: '+str(round(scharr_row['std_kernel_ms'],5)):>12} {'std: '+str(round(sobel_row['std_total_ms'],5)):>12} {'std: '+str(round(scharr_row['std_total_ms'],5)):>12}")
    print(f"{'':>8} {'min: '+str(round(sobel_row['min_kernel_ms'],5)):>12} {'min: '+str(round(scharr_row['min_kernel_ms'],5)):>12} {'min: '+str(round(sobel_row['min_total_ms'],5)):>12} {'min: '+str(round(scharr_row['min_total_ms'],5)):>12}")
    print(f"{'':>8} {'max: '+str(round(sobel_row['max_kernel_ms'],5)):>12} {'max: '+str(round(scharr_row['max_kernel_ms'],5)):>12} {'max: '+str(round(sobel_row['max_total_ms'],5)):>12} {'max: '+str(round(scharr_row['max_total_ms'],5)):>12}")
    winner_k = 'Scharr' if scharr_row['avg_kernel_ms'] < sobel_row['avg_kernel_ms'] else 'Sobel'
    winner_t = 'Scharr' if scharr_row['avg_total_ms'] < sobel_row['avg_total_ms'] else 'Sobel'
    print(f"  -> Kernel winner: {winner_k}  |  Total path winner: {winner_t}")


In [ ]:
\
print("=" * 80)
print("TABLE 2: Naive CUDA Sobel vs Naive CUDA Scharr")
print("=" * 80)

naive_df = results_df[results_df['implementation'] == 'naive_cuda'][['size','operator','avg_kernel_ms','std_kernel_ms','min_kernel_ms','max_kernel_ms','avg_total_ms','std_total_ms','min_total_ms','max_total_ms']].copy()

for size_label in ['128x128', '512x512', '1280x720', '1920x1080']:
    sobel_row = naive_df[(naive_df['size'] == size_label) & (naive_df['operator'] == 'sobel')].iloc[0]
    scharr_row = naive_df[(naive_df['size'] == size_label) & (naive_df['operator'] == 'scharr')].iloc[0]
    kernel_ratio = scharr_row['avg_kernel_ms'] / sobel_row['avg_kernel_ms']
    total_ratio = scharr_row['avg_total_ms'] / sobel_row['avg_total_ms']
    print(f"\n{size_label}  (Scharr/Sobel kernel ratio: {kernel_ratio:.4f}, total ratio: {total_ratio:.4f})")
    print(f"{'':>12} {'sobel':>12} {'scharr':>12} {'sobel':>12} {'scharr':>12}")
    print(f"{'':>12} {'kernel ms':>12} {'kernel ms':>12} {'total ms':>12} {'total ms':>12}")
    print(f"{'':>12} {sobel_row['avg_kernel_ms']:>12.5f} {scharr_row['avg_kernel_ms']:>12.5f} {sobel_row['avg_total_ms']:>12.5f} {scharr_row['avg_total_ms']:>12.5f}")
    print(f"{'':>12} {'std: '+str(round(sobel_row['std_kernel_ms'],5)):>12} {'std: '+str(round(scharr_row['std_kernel_ms'],5)):>12} {'std: '+str(round(sobel_row['std_total_ms'],5)):>12} {'std: '+str(round(scharr_row['std_total_ms'],5)):>12}")
    print(f"{'':>12} {'min: '+str(round(sobel_row['min_kernel_ms'],5)):>12} {'min: '+str(round(scharr_row['min_kernel_ms'],5)):>12} {'min: '+str(round(sobel_row['min_total_ms'],5)):>12} {'min: '+str(round(scharr_row['min_total_ms'],5)):>12}")
    print(f"{'':>12} {'max: '+str(round(sobel_row['max_kernel_ms'],5)):>12} {'max: '+str(round(scharr_row['max_kernel_ms'],5)):>12} {'max: '+str(round(sobel_row['max_total_ms'],5)):>12} {'max: '+str(round(scharr_row['max_total_ms'],5)):>12}")
    winner_k = 'Scharr' if scharr_row['avg_kernel_ms'] < sobel_row['avg_kernel_ms'] else 'Sobel'
    winner_t = 'Scharr' if scharr_row['avg_total_ms'] < sobel_row['avg_total_ms'] else 'Sobel'
    print(f"  -> Kernel winner: {winner_k}  |  Total path winner: {winner_t}")


In [ ]:
\
print("=" * 80)
print("TABLE 3: Optimized CUDA Sobel vs Optimized CUDA Scharr")
print("=" * 80)

opt_df = results_df[results_df['implementation'] == 'optimized_cuda'][['size','operator','avg_kernel_ms','std_kernel_ms','min_kernel_ms','max_kernel_ms','avg_total_ms','std_total_ms','min_total_ms','max_total_ms']].copy()

for size_label in ['128x128', '512x512', '1280x720', '1920x1080']:
    sobel_row = opt_df[(opt_df['size'] == size_label) & (opt_df['operator'] == 'sobel')].iloc[0]
    scharr_row = opt_df[(opt_df['size'] == size_label) & (opt_df['operator'] == 'scharr')].iloc[0]
    kernel_ratio = scharr_row['avg_kernel_ms'] / sobel_row['avg_kernel_ms']
    total_ratio = scharr_row['avg_total_ms'] / sobel_row['avg_total_ms']
    print(f"\n{size_label}  (Scharr/Sobel kernel ratio: {kernel_ratio:.4f}, total ratio: {total_ratio:.4f})")
    print(f"{'':>14} {'sobel':>12} {'scharr':>12} {'sobel':>12} {'scharr':>12}")
    print(f"{'':>14} {'kernel ms':>12} {'kernel ms':>12} {'total ms':>12} {'total ms':>12}")
    print(f"{'':>14} {sobel_row['avg_kernel_ms']:>12.5f} {scharr_row['avg_kernel_ms']:>12.5f} {sobel_row['avg_total_ms']:>12.5f} {scharr_row['avg_total_ms']:>12.5f}")
    print(f"{'':>14} {'std: '+str(round(sobel_row['std_kernel_ms'],5)):>12} {'std: '+str(round(scharr_row['std_kernel_ms'],5)):>12} {'std: '+str(round(sobel_row['std_total_ms'],5)):>12} {'std: '+str(round(scharr_row['std_total_ms'],5)):>12}")
    print(f"{'':>14} {'min: '+str(round(sobel_row['min_kernel_ms'],5)):>12} {'min: '+str(round(scharr_row['min_kernel_ms'],5)):>12} {'min: '+str(round(sobel_row['min_total_ms'],5)):>12} {'min: '+str(round(scharr_row['min_total_ms'],5)):>12}")
    print(f"{'':>14} {'max: '+str(round(sobel_row['max_kernel_ms'],5)):>12} {'max: '+str(round(scharr_row['max_kernel_ms'],5)):>12} {'max: '+str(round(sobel_row['max_total_ms'],5)):>12} {'max: '+str(round(scharr_row['max_total_ms'],5)):>12}")
    winner_k = 'Scharr' if scharr_row['avg_kernel_ms'] < sobel_row['avg_kernel_ms'] else 'Sobel'
    winner_t = 'Scharr' if scharr_row['avg_total_ms'] < sobel_row['avg_total_ms'] else 'Sobel'
    print(f"  -> Kernel winner: {winner_k}  |  Total path winner: {winner_t}")


In [ ]:
\
print("=" * 80)
print("SUMMARY: Scharr vs Sobel Speedup (Scharr/Sobel ratio, <1 means Sobel wins)")
print("=" * 80)

summary_rows = []
for impl in ['cpu', 'naive_cuda', 'optimized_cuda']:
    impl_df = results_df[results_df['implementation'] == impl]
    for size_label in ['128x128', '512x512', '1280x720', '1920x1080']:
        s = impl_df[(impl_df['size'] == size_label) & (impl_df['operator'] == 'sobel')].iloc[0]
        c = impl_df[(impl_df['size'] == size_label) & (impl_df['operator'] == 'scharr')].iloc[0]
        summary_rows.append({
            'implementation': impl,
            'size': size_label,
            'sobel_kernel_ms': s['avg_kernel_ms'],
            'scharr_kernel_ms': c['avg_kernel_ms'],
            'scharr_sobel_kernel_ratio': c['avg_kernel_ms'] / s['avg_kernel_ms'],
            'sobel_total_ms': s['avg_total_ms'],
            'scharr_total_ms': c['avg_total_ms'],
            'scharr_sobel_total_ratio': c['avg_total_ms'] / s['avg_total_ms'],
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

print()
for impl in ['cpu', 'naive_cuda', 'optimized_cuda']:
    impl_s = summary_df[summary_df['implementation'] == impl]
    avg_k_ratio = impl_s['scharr_sobel_kernel_ratio'].mean()
    avg_t_ratio = impl_s['scharr_sobel_total_ratio'].mean()
    print(f"{impl:>18}: avg kernel ratio = {avg_k_ratio:.4f}  |  avg total ratio = {avg_t_ratio:.4f}")
    if avg_k_ratio < 1.0:
        print(f"{'':>18}  -> Scharr is faster on kernel (avg {abs(1-avg_k_ratio)*100:.2f}% faster)")
    else:
        print(f"{'':>18}  -> Sobel is faster on kernel (avg {abs(1-avg_k_ratio)*100:.2f}% faster)")


In [ ]:
\
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 2, figsize=(16, 18))

for row_idx, impl in enumerate(['cpu', 'naive_cuda', 'optimized_cuda']):
    impl_df = results_df[results_df['implementation'] == impl].copy()

    ax_k = axes[row_idx, 0]
    ax_t = axes[row_idx, 1]

    for op, color, marker in [('sobel', 'tab:blue', 'o'), ('scharr', 'tab:orange', 's')]:
        sub = impl_df[impl_df['operator'] == op].sort_values('pixels')
        ax_k.plot(sub['pixels'], sub['avg_kernel_ms'], marker=marker, color=color, label=op.capitalize())
        ax_t.plot(sub['pixels'], sub['avg_total_ms'], marker=marker, color=color, label=op.capitalize())

    impl_label = {'cpu': 'CPU', 'naive_cuda': 'Naive CUDA', 'optimized_cuda': 'Optimized CUDA'}[impl]
    ax_k.set_title(f'{impl_label}: Avg Kernel Time')
    ax_k.set_xlabel('Pixels')
    ax_k.set_ylabel('Kernel time (ms)')
    ax_k.set_xscale('log')
    ax_k.grid(True, alpha=0.3)
    ax_k.legend()

    ax_t.set_title(f'{impl_label}: Avg Total Path Time')
    ax_t.set_xlabel('Pixels')
    ax_t.set_ylabel('Total time (ms)')
    ax_t.set_xscale('log')
    ax_t.grid(True, alpha=0.3)
    ax_t.legend()

plt.tight_layout()
plt.show()

# Scharr/Sobel ratio plot
fig2, ax2 = plt.subplots(figsize=(12, 6))
for impl, style in [('cpu', '-o'), ('naive_cuda', '--s'), ('optimized_cuda', ':^')]:
    s = summary_df[summary_df['implementation'] == impl]
    ax2.plot(s['size'], s['scharr_sobel_kernel_ratio'], style, label=f'{impl} kernel ratio')
    ax2.plot(s['size'], s['scharr_sobel_total_ratio'], style.replace('o','x').replace('s','+').replace('^','*'),
             label=f'{impl} total ratio', alpha=0.5)
ax2.axhline(y=1.0, color='black', linestyle='--', linewidth=1, label='Equal (ratio=1.0)')
ax2.set_title('Scharr/Sobel Time Ratio by Implementation and Image Size\n(<1 = Scharr faster, >1 = Sobel faster)')
ax2.set_xlabel('Image size')
ax2.set_ylabel('Scharr/Sobel ratio')
ax2.grid(True, alpha=0.3)
ax2.legend()
plt.tight_layout()
plt.show()
